# Computing Eigenvalues and Eigenvectors

## Introduction

In [linear algebra notes](../basic_knowledges/linear_algebra.md), we talked about the concepts of eigenvalues and eigenvectors, which should be clear enough to apply some algorithms. But first of all, let's talk about the conditioning of eigenvalue and eigenvector problem.

We will start with the definition of the eigenvalue and eigenvector, with some small disturbance for each term:

$$(\textbf{A}+\delta\textbf{A})(\textbf{v}_i+\delta\textbf{v}_i)=(\lambda_i+\delta\lambda_i)(\textbf{v}_i+\delta\textbf{v}_i)$$

By expanding these terms, we will finally get:

$$\mathbf{A} \delta \mathbf{v}_i + \delta \mathbf{A} \mathbf{v}_i \approx \lambda_i \delta \mathbf{v}_i + \delta \lambda_i \mathbf{v}_i$$

It will be more convenient to turn these vector terms to the scalar terms. Hence, we can multiply the left eigenvalue with Hermitian transpose, $\textbf{w}^H$, to it:

$$
\begin{aligned}
\mathbf{w}_i^{\text{H}} \mathbf{A} \delta \mathbf{v}_i + \mathbf{w}_i^{\text{H}} \delta \mathbf{A} \mathbf{v}_i & \approx \lambda_i \mathbf{w}_i^{\text{H}} \delta \mathbf{v}_i + \delta \lambda_i \mathbf{w}_i^{\text{H}} \mathbf{v}_i, \\
\mathbf{w}_i^{\text{H}} \delta \mathbf{A} \mathbf{v}_i & \approx \delta \lambda_i \mathbf{w}_i^{\text{H}} \mathbf{v}_i, \\
\delta \lambda_i & \approx \frac{\mathbf{w}_i^{\text{H}} \delta \mathbf{A} \mathbf{v}_i}{\mathbf{w}_i^{\text{H}} \mathbf{v}_i}.
\end{aligned}
$$

Then by applying the inequality identity, we will finally have:

$$
\begin{aligned}
|\delta \lambda_i| & \approx \frac{|\mathbf{w}_i^{\text{H}} \delta \mathbf{A} \mathbf{v}_i|}{|\mathbf{w}_i^{\text{H}} \mathbf{v}_i|} \\
& \le \frac{\|\mathbf{w}_i\|_2 \|\delta \mathbf{A} \mathbf{v}_i\|_2}{|\mathbf{w}_i^{\text{H}} \mathbf{v}_i|} \\
& \le \frac{\|\mathbf{w}_i\|_2 \|\mathbf{v}_i\|_2}{|\mathbf{w}_i^{\text{H}} \mathbf{v}_i|} \|\delta \mathbf{A}\|_2 \\
& = \frac{1}{\cos \theta_i} \|\delta \mathbf{A}\|_2
\end{aligned}
$$

Where $\theta\in(-\frac{\pi}{2},\frac{\pi}{2}]$, then we can find that the angle between left and right eigenvectors determines the eigenvalue sensitivity. For real symmetry matrix, these two eigenvectors are in the same direction, and it has the least sensitivity.

## Four eigenvalue-eigenvector transformations

<center>

| Transformation | Eigenvalues | Eigenvectors | Purpose |
| :--- | :--- | :--- | :--- |
| Shift $A - \sigma I$ | $\lambda_i - \sigma$ | unchanged | move spectrum |
| Inverse $A^{-1}$ | $1/\lambda_i$ | unchanged | amplify small eigenvalues |
| Power $A^k$ | $\lambda_i^k$ | unchanged | separate magnitudes |
| Similarity $T^{-1}AT$ | unchanged | transformed | numerical stability |

</center>

## Power Iteration


As we learned in [linear algebra notes](../basic_knowledges/linear_algebra.md), the eigenvectors and eigenvalues describe how a vector is transformed under a change in the length or direction of the basis. The power iteration is easy, it just repeats $\textbf{x}_k=\textbf{Ax}_{k-1}$. This process will cause the vector's direction to turn toward the direction with the largest eigenvalue (since the matrix will magnify this direction the most). Hence, we can use this method to find the eigenvector with the largest eigenvalue.

We can also find the eigenvector with the smallest eigenvalue by doing the inverse, since all eigenvalues are inverted if the matrix is inverted. We can solve $\textbf{Ay}_k=\textbf{x}_{k-1}$ for $\textbf{y}_k$ to complete the iteration.

Note that for each iteration, we will normalize the vector to make the process numerically stable. Since this algorithm is not that complicated, I will not show the code here.

## Shifting and Rayleigh Quotient Iteration

We can accelerate the iteration process by doing the shifting. Here, we introduce the concept of the Rayleigh quotient, which will give us a different approach to the eigenvalue:

$$
\begin{aligned}
\textbf{v}_i^H\textbf{Av}_i &= \lambda\textbf{v}_i^H\textbf{v}_i \\
\lambda_i &= \frac{\textbf{v}_i^H\textbf{Av}_i}{\textbf{v}_i^H\textbf{v}_i}
\end{aligned}
$$

We usually don't know the eigenvalue initially, so we start with a guess $\textbf{x}_0$. By replacing $\textbf{x}_i$ with $\textbf{v}_i$, we can establish the iteration algorithm. The numerator of this quotient is the vector product (or, in this case, the vector 2-norm square) in $\textbf{A}$-coordinates, and the denominator is the one in the Euclidean coordinates. We can also consider this as the weighted average of the components of the vector under each eigenvector. Since the input vector is normalized, this "weighted average" will be close to the eigenvalue that the corresponding eigenvector has the smallest angle with $\textbf{x}_0$.

The Rayleigh quotient iteration will be based on this: by doing $\textbf{A}-\sigma_k\textbf{I}$, our eigenvalue is shifted each iteration, effectively accelerating convergence if we are going to find the smallest eigenvalue.

## Orthogonal Iteration

Can we calculate the eigenvector simultaneously? The answer is yes, by using the orthogonal iteration. The orthogonal iteration is based on the power iteration. Still, it uses the QR factorization to make all input vectors independent of each other, meaning that even if the corresponding eigenvalue is not the largest, the QR factorization helps extract the direction with the larger eigenvalue from the current vector. Note that due to the property of the non-symmetric matrix, the eigenvectors will not be orthogonal to each other, hence the orthogonal iteration will not directly give the correct answer (but will still provide some information, which will be mentioned later).

In [5]:
# Orthogonal Iteration
import numpy as np

def orth_iter(A, eps_tot=1e-8, max_iter=1000):
    X = np.eye(A.shape[0])
    I = np.eye(A.shape[0])
    iteration = 0
    error = 1.0
    
    Q_hat_prev = np.zeros(A.shape)

    while error > eps_tot and iteration < max_iter:
        Q_hat, R = np.linalg.qr(X)
        X = A @ Q_hat

        error = np.linalg.norm(I - abs(Q_hat_prev.T @ Q_hat))
        Q_hat_prev = Q_hat.copy()
        iteration += 1
        
    V = Q_hat
    return V, iteration

In [12]:
A = np.array([
    [1, 2, 0],
    [2, 3, 1],
    [0, 1, 5]
], dtype=float)

X, n = orth_iter(A)

val_np, X_np = np.linalg.eig(A)
X_np = X_np[:, np.argsort(val_np[::-1])]

print("Eigenvector calculated by the orthogonal iteration:")
print(X)
print("Eigenvector calculated numpy function:")
print(X_np)
print(f"Iteration: {n}")


Eigenvector calculated by the orthogonal iteration:
[[-0.21493528 -0.50489607 -0.8359921 ]
 [-0.49265589 -0.6830536   0.53919195]
 [-0.8432633   0.5277478  -0.1019277 ]]
Eigenvector calculated numpy function:
[[ 0.21493528 -0.50489607 -0.8359921 ]
 [ 0.49265588 -0.6830536   0.53919195]
 [ 0.84326331  0.52774779 -0.1019277 ]]
Iteration: 48


## QR Iteration